In [5]:
# Core data manipulation
import pandas as pd
import numpy as np
from numpy import polyfit, poly1d

# Visualization libraries
import matplotlib.pyplot as plt
import seaborn as sns
import altair as alt

In [10]:
owid = pd.read_csv('owid-covid-data.csv')
display(owid.head())

,iso_code,continent,location,date,total_cases,new_cases,new_cases_smoothed,total_deaths,new_deaths,new_deaths_smoothed,...,male_smokers,handwashing_facilities,hospital_beds_per_thousand,life_expectancy,human_development_index,population,excess_mortality_cumulative_absolute,excess_mortality_cumulative,excess_mortality,excess_mortality_cumulative_per_million
0,AFG,Asia,Afghanistan,2020-01-05,0.0,0.0,NaN,0.0,0.0,NaN,...,NaN,37.75,0.5,64.83,0.51,41128772,NaN,NaN,NaN,NaN
1,AFG,Asia,Afghanistan,2020-01-06,0.0,0.0,NaN,0.0,0.0,NaN,...,NaN,37.75,0.5,64.83,0.51,41128772,NaN,NaN,NaN,NaN
2,AFG,Asia,Afghanistan,2020-01-07,0.0,0.0,NaN,0.0,0.0,NaN,...,NaN,37.75,0.5,64.83,0.51,41128772,NaN,NaN,NaN,NaN
3,AFG,Asia,Afghanistan,2020-01-08,0.0,0.0,NaN,0.0,0.0,NaN,...,NaN,37.75,0.5,64.83,0.51,41128772,NaN,NaN,NaN,NaN
4,AFG,Asia,Afghanistan,2020-01-09,0.0,0.0,NaN,0.0,0.0,NaN,...,NaN,37.75,0.5,64.83,0.51,41128772,NaN,NaN,NaN,NaN


In [15]:
owid['stringency_index']

0         0.0
1         0.0
2         0.0
3         0.0
4         0.0
         ... 
429430    NaN
429431    NaN
429432    NaN
429433    NaN
429434    NaN
Name: stringency_index, Length: 429435, dtype: float64

##### Visualization #3

In [11]:
# setting date to date time format
owid['date'] = pd.to_datetime(owid['date'])

# ---- 1️⃣ First confirmed death per country ----
first_death = (
    owid[owid['total_deaths'] > 0]
    .groupby('location')['date']
    .min()
    .reset_index()
    .rename(columns={'date': 'first_death_date'})
)

# ---- 2️⃣ First strong restriction date (stringency ≥ 70) ----
strong_restriction = (
    owid[owid['stringency_index'] >= 70]
    .groupby('location')['date']
    .min()
    .reset_index()
    .rename(columns={'date': 'restriction_date'})
)

# ---- 3️⃣ Merge these together ----
country_dates = pd.merge(first_death, strong_restriction, on='location', how='inner')

# ---- 4️⃣ Calculate days between ----
country_dates['days_to_restrictions'] = (
    country_dates['restriction_date'] - country_dates['first_death_date']
).dt.days

# ---- 5️⃣ Get deaths per million (use max value per country) ----
deaths_pm = (
    owid.groupby('location')['total_deaths_per_million']
    .max()
    .reset_index()
)

# ---- 6️⃣ Get continent + population ----
meta = (
    owid.groupby('location')
    .agg({
        'continent': 'first',
        'population': 'first'
    })
    .reset_index()
)

# ---- 7️⃣ Final dataset ----
country_owid = (
    country_dates
    .merge(deaths_pm, on='location')
    .merge(meta, on='location')
)

# Drop missing continent rows (e.g., World, income groups)
country_owid = country_owid[country_owid['continent'].notna()]

country_owid.head()

,location,first_death_date,restriction_date,days_to_restrictions,total_deaths_per_million,continent,population
0,Afghanistan,2020-03-29,2020-04-05,7,197.10,Asia,41128772
1,Albania,2020-03-15,2020-03-13,-2,1274.93,Europe,2842318
2,Algeria,2020-03-29,2020-03-23,-6,151.31,Africa,44903228
3,Angola,2020-05-24,2020-03-27,-58,54.36,Africa,35588996
4,Argentina,2020-03-08,2020-03-19,11,2877.54,South America,45510324


In [17]:
country_owid = country_owid[
    country_owid['days_to_restrictions'].notna() &
    country_owid['total_deaths_per_million'].notna() &
    country_owid['population'].notna()
]

In [21]:
country_owid[['days_to_restrictions',
            'total_deaths_per_million',
            'population']].isna().sum()

days_to_restrictions        0
total_deaths_per_million    0
population                  0
dtype: int64

In [23]:
alt.data_transformers.disable_max_rows()

scatter = alt.Chart(country_owid).mark_circle(opacity=0.7).encode(
    x=alt.X('days_to_restrictions:Q',
            title='Days Between First Death and Strong Restrictions'),
    y=alt.Y('total_deaths_per_million:Q',
            title='COVID Deaths per Million'),
    color=alt.Color('continent:N', title='Continent'),
    size=alt.Size('population:Q',
                  scale=alt.Scale(type='sqrt', range=[30, 2000]),
                  legend=alt.Legend(title='Population')),
    tooltip=['location', 'continent',
             'days_to_restrictions',
             'total_deaths_per_million',
             'population']
).properties(
    width=700,
    height=500,
    title='Policy Timing and COVID Mortality'  
)

# regression = alt.Chart(country_owid).transform_regression(
#     'days_to_restrictions',
#     'total_deaths_per_million'
# ).mark_line(color='black')

# chart = (scatter + regression).properties(
#     width=700,
#     height=500,
#     title='Policy Timing and COVID Mortality'
# )

scatter

alt.Chart(...)

In [18]:
alt.data_transformers.disable_max_rows()

scatter = alt.Chart(country_owid).mark_circle(opacity=0.7).encode(
    x=alt.X(
        'days_to_restrictions:Q',
        title='Days Between First Death and Strong Restrictions'
    ),
    y=alt.Y(
        'total_deaths_per_million:Q',
        title='COVID Deaths per Million'
    ),
    color=alt.Color('continent:N', title='Continent'),
    size=alt.Size(
        'population:Q',
        scale=alt.Scale(range=[50, 3000]),
        title='Population'
    ),
    tooltip=[
        'location',
        'continent',
        'days_to_restrictions',
        'total_deaths_per_million',
        'population'
    ]
)

# Regression line
regression = alt.Chart(country_owid).transform_regression(
    'days_to_restrictions',
    'total_deaths_per_million'
).mark_line(color='black')

# Label important countries
important = ['United States', 'Italy', 'China', 'Brazil', 'India']

labels = alt.Chart(
    country_owid[country_owid['location'].isin(important)]
).mark_text(
    align='left',
    dx=7,
    dy=-5
).encode(
    x='days_to_restrictions:Q',
    y='total_deaths_per_million:Q',
    text='location'
)

chart = (scatter + regression + labels).properties(
    width=700,
    height=500,
    title='Policy Timing and COVID Mortality'
)

chart

alt.LayerChart(...)